# Data Cleaning: OSMI Mental Health in Tech Survey (2014)
Cleans the raw survey into `data/processed/survey_clean.csv` for the dashboard, analysis, and model. Each section explains what was wrong and why it was fixed the way it was.

## 1. Load the data
Loading the 2014 OSMI Mental Health in Tech survey and getting a first look at its size, columns, and data types.

In [2]:
import pandas as pd
import numpy as np


df = pd.read_csv("../data/raw/survey.csv")

print(df.shape)
df.info()
df.head()

(1259, 27)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Timestamp                  1259 non-null   object
 1   Age                        1259 non-null   int64 
 2   Gender                     1259 non-null   object
 3   Country                    1259 non-null   object
 4   state                      744 non-null    object
 5   self_employed              1241 non-null   object
 6   family_history             1259 non-null   object
 7   treatment                  1259 non-null   object
 8   work_interfere             995 non-null    object
 9   no_employees               1259 non-null   object
 10  remote_work                1259 non-null   object
 11  tech_company               1259 non-null   object
 12  benefits                   1259 non-null   object
 13  care_options               1259 non-null   object
 1

,Timestamp,Age,Gender,Country,state,self_employed,family_history,treatment,work_interfere,no_employees,...,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
0,2014-08-27 11:29:31,37,Female,United States,IL,NaN,No,Yes,Often,6-25,...,Somewhat easy,No,No,Some of them,Yes,No,Maybe,Yes,No,NaN
1,2014-08-27 11:29:37,44,M,United States,IN,NaN,No,No,Rarely,More than 1000,...,Don't know,Maybe,No,No,No,No,No,Don't know,No,NaN
2,2014-08-27 11:29:44,32,Male,Canada,NaN,NaN,No,No,Rarely,6-25,...,Somewhat difficult,No,No,Yes,Yes,Yes,Yes,No,No,NaN
3,2014-08-27 11:29:46,31,Male,United Kingdom,NaN,NaN,Yes,Yes,Often,26-100,...,Somewhat difficult,Yes,Yes,Some of them,No,Maybe,Maybe,No,Yes,NaN
4,2014-08-27 11:30:22,31,Male,United States,TX,NaN,No,No,Never,100-500,...,Don't know,No,No,Some of them,Yes,Yes,Yes,Don't know,No,NaN


## 2. Missing values
Checking which columns have missing data and how much, to decide how to handle each one.

In [3]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(1)

pd.DataFrame({"missing": missing, "percent": missing_pct})[missing > 0].sort_values("missing", ascending=False)

,missing,percent
comments,1095,87.0
state,515,40.9
work_interfere,264,21.0
self_employed,18,1.4


**Findings:** `comments` is mostly empty (optional free text). `state` is likely only filled in for US respondents. `work_interfere` blanks probably mean the respondent had no condition. `self_employed` is missing for 18 rows. Each is handled in its own section below.

## 3. Age
Checking that ages fall in a realistic range for working adults.

In [4]:
df["Age"].describe()

count    1.259000e+03
mean     7.942815e+07
std      2.818299e+09
min     -1.726000e+03
25%      2.700000e+01
50%      3.100000e+01
75%      3.600000e+01
max      1.000000e+11
Name: Age, dtype: float64

In [5]:
df[(df["Age"] < 18) | (df["Age"] > 75)]["Age"].sort_values()

715           -1726
143             -29
1127             -1
734               5
989               8
1090             11
364             329
390     99999999999
Name: Age, dtype: int64

In [6]:
# Treat anything outside 18–75 as a bad entry
bad_age = (df["Age"] < 18) | (df["Age"] > 75)
print(f"Invalid ages: {bad_age.sum()}")

df.loc[bad_age, "Age"] = np.nan
df["Age"] = df["Age"].fillna(df["Age"].median()).astype(int)

df["Age"].describe()

Invalid ages: 8


count    1259.000000
mean       32.069897
std         7.265565
min        18.000000
25%        27.000000
50%        31.000000
75%        36.000000
max        72.000000
Name: Age, dtype: float64

**Decision:** 8 ages were clearly invalid (negative, 5–11, 329, and 99999999999). Used 18–75 as a plausible range for working adults; no real ages fall near either cutoff, so the exact bounds don't change which rows are flagged. Replaced invalid ages with the median (31) to keep those respondents' other answers.

## 4. Gender
`Gender` was a free-text field, so the same answer appears in many forms. Looking at every distinct response before grouping them.

In [7]:
df["Gender"].value_counts()

Gender
Male                                              615
male                                              206
Female                                            121
M                                                 116
female                                             62
F                                                  38
m                                                  34
f                                                  15
Make                                                4
Male                                                3
Woman                                               3
Cis Male                                            2
Man                                                 2
Female                                              2
Female (trans)                                      2
Male-ish                                            1
Trans-female                                        1
Male (CIS)                                          1
Mal                  

In [8]:
print(df["Gender"].str.strip().str.lower().value_counts().to_string())

Gender
male                                              824
female                                            185
m                                                 150
f                                                  53
woman                                               4
make                                                4
cis male                                            3
man                                                 2
female (trans)                                      2
something kinda male?                               1
cis female                                          1
trans-female                                        1
maile                                               1
male-ish                                            1
queer/she/they                                      1
male (cis)                                          1
mal                                                 1
nah                                                 1
all                  

In [9]:
# Save the original answers only the first time, so re-running is safe
if "Gender_raw" not in df.columns:
    df["Gender_raw"] = df["Gender"]

g = df["Gender_raw"].str.strip().str.lower()

male = {"male", "m", "man", "cis male", "cis man", "male (cis)",
        "make", "mal", "maile", "mail", "malr", "msle"}
female = {"female", "f", "woman", "cis female", "female (cis)", "cis-female/femme",
          "femake", "femail", "female (trans)", "trans woman", "trans-female"}
unknown = {"a little about you", "p", "all", "nah"}

def group_gender(x):
    if x in male:
        return "Male"
    if x in female:
        return "Female"
    if x in unknown:
        return "Unknown"
    return "Non-binary/Other"

df["Gender"] = g.map(group_gender)

df["Gender"].value_counts()

Gender
Male                990
Female              251
Non-binary/Other     14
Unknown               4
Name: count, dtype: int64

   **Decision:** Grouped ~40 free-text answers into Male, Female, Non-binary/Other, and Unknown. Typos were matched to the intended answer; trans women grouped as Female per their stated gender; qualified answers like "male-ish" grouped as Non-binary/Other; non-answers marked Unknown. Original responses kept in `Gender_raw`.

## 5. Country and state
Checking country spread and whether `state` is only filled in for US respondents.

In [10]:
df["Country"].value_counts().head(15)

Country
United States     751
United Kingdom    185
Canada             72
Germany            45
Netherlands        27
Ireland            27
Australia          21
France             13
India              10
New Zealand         8
Poland              7
Italy               7
Sweden              7
Switzerland         7
South Africa        6
Name: count, dtype: int64

In [11]:
is_us = df["Country"] == "United States"
pd.crosstab(is_us, df["state"].isna(), rownames=["US?"], colnames=["state missing?"])

state missing?,False,True
US?,,
False,4,504
True,740,11


In [12]:
df.loc[~is_us & df["state"].notna(), ["Country", "state"]]

,Country,state
319,Latvia,NY
488,Israel,MD
989,"Bahamas, The",IL
1179,Bulgaria,UT


In [13]:
df.loc[~is_us, "state"] = "Non-US"                          
df.loc[is_us & df["state"].isna(), "state"] = "Unknown"    
print(df["state"].isna().sum())   
df["state"].value_counts().head()

0


state
Non-US    508
CA        138
WA         70
NY         56
TN         45
Name: count, dtype: int64

In [14]:
counts = df["Country"].value_counts()
df["Country_grouped"] = df["Country"].where(df["Country"].map(counts) >= 20, "Other")

df["Country_grouped"].value_counts()

Country_grouped
United States     751
United Kingdom    185
Other             131
Canada             72
Germany            45
Netherlands        27
Ireland            27
Australia          21
Name: count, dtype: int64

**Decision:** `state` only applies to US respondents. Set it to "Non-US" for non-US rows (including 4 that had a state filled in by mistake) and "Unknown" for 11 US respondents who skipped it. Added `Country_grouped`, which keeps countries with 20+ respondents and groups the rest as "Other"; the original `Country` column is unchanged for the dashboard map.

## 6. Remaining missing values
Working out why `self_employed`, `work_interfere`, and `comments` are blank before choosing a fix.

In [15]:
df.loc[df["self_employed"].isna(), "Timestamp"]

0     2014-08-27 11:29:31
1     2014-08-27 11:29:37
2     2014-08-27 11:29:44
3     2014-08-27 11:29:46
4     2014-08-27 11:30:22
5     2014-08-27 11:31:22
6     2014-08-27 11:31:50
7     2014-08-27 11:32:05
8     2014-08-27 11:32:39
9     2014-08-27 11:32:43
10    2014-08-27 11:32:44
11    2014-08-27 11:32:49
12    2014-08-27 11:33:23
13    2014-08-27 11:33:26
14    2014-08-27 11:33:57
15    2014-08-27 11:34:00
16    2014-08-27 11:34:20
17    2014-08-27 11:34:37
Name: Timestamp, dtype: object

In [16]:
pd.crosstab(df["work_interfere"].fillna("BLANK"), df["treatment"])

treatment,No,Yes
work_interfere,,
BLANK,260,4
Never,183,30
Often,21,123
Rarely,51,122
Sometimes,107,358


In [17]:
df["comments"].dropna().head(10)

13    I'm not on my company's health insurance which...
15    I have chronic low-level neurological issues t...
16    My company does provide healthcare but not to ...
24                  Relatively new job. Ask again later
25    Sometimes I think  about using drugs for my me...
33    I selected my current employer based on its po...
45    Our health plan has covered my psychotherapy a...
49    I just started a new job last week hence a lot...
59    In addition to my own mental health issues I'v...
66                      Thanks for doing this research.
Name: comments, dtype: object

In [18]:
df["self_employed"].value_counts()

self_employed
No     1095
Yes     146
Name: count, dtype: int64

In [21]:
df["self_employed"] = df["self_employed"].fillna("Unknown")
df["work_interfere"] = df["work_interfere"].fillna("Unknown")
df = df.drop(columns=["comments"], errors="ignore")

df.isna().sum().sum()

np.int64(0)

**Decision:**
- `self_employed`: the 18 blanks are the first 18 responses (first 5 minutes of the survey), so the question was likely added after launch. Labeled "Unknown" rather than guessing an answer.
- `work_interfere`: the question only applies to people with a mental health condition, and most blanks didn't seek treatment (260 of 264). But a blank doesn't confirm someone has no condition (4 blanks did seek treatment), so labeled "Unknown" rather than "Not applicable".
- `comments`: dropped. Optional free text, 87% empty, not usable for analysis. Still available in the raw CSV.

**Heads-up for modeling:** `work_interfere` almost directly reveals `treatment` (Unknown ≈ no treatment, "Often" ≈ treatment). Including it in a model predicting `treatment` will inflate accuracy without teaching us much. Worth discussing.

## 7. Final checks and save
Confirming categories are consistent, checking for duplicates and dates, then saving the clean dataset.

In [19]:
skip = ["Timestamp", "Gender_raw", "Country", "state"]
for col in df.select_dtypes("object").columns.drop(skip, errors="ignore"):
    print(f"{col}: {sorted(df[col].unique())}")

Gender: ['Female', 'Male', 'Non-binary/Other', 'Unknown']
self_employed: ['No', 'Yes']
family_history: ['No', 'Yes']
treatment: ['No', 'Yes']
work_interfere: ['Never', 'Not applicable', 'Often', 'Rarely', 'Sometimes']
no_employees: ['1-5', '100-500', '26-100', '500-1000', '6-25', 'More than 1000']
remote_work: ['No', 'Yes']
tech_company: ['No', 'Yes']
benefits: ["Don't know", 'No', 'Yes']
care_options: ['No', 'Not sure', 'Yes']
wellness_program: ["Don't know", 'No', 'Yes']
seek_help: ["Don't know", 'No', 'Yes']
anonymity: ["Don't know", 'No', 'Yes']
leave: ["Don't know", 'Somewhat difficult', 'Somewhat easy', 'Very difficult', 'Very easy']
mental_health_consequence: ['Maybe', 'No', 'Yes']
phys_health_consequence: ['Maybe', 'No', 'Yes']
coworkers: ['No', 'Some of them', 'Yes']
supervisor: ['No', 'Some of them', 'Yes']
mental_health_interview: ['Maybe', 'No', 'Yes']
phys_health_interview: ['Maybe', 'No', 'Yes']
mental_vs_physical: ["Don't know", 'No', 'Yes']
obs_consequence: ['No', 'Yes'

In [20]:
print("Duplicate rows:", df.duplicated().sum())

df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df["Timestamp"].dt.year.value_counts().sort_index()

Duplicate rows: 0


Timestamp
2014    1190
2015      68
2016       1
Name: count, dtype: int64

In [21]:
from pathlib import Path

out = Path("../data/processed")
out.mkdir(parents=True, exist_ok=True)
df.to_csv(out / "survey_clean.csv", index=False)

print(df.shape)   

(1259, 28)


## Summary
Clean data saved to `data/processed/survey_clean.csv`: 1,259 rows, 28 columns, no missing values, no rows dropped.

| Column | Fix |
|---|---|
| `Age` | 8 invalid values (negative, under 18, 329, 99999999999) replaced with median (31) |
| `Gender` | ~40 free-text answers grouped into Male / Female / Non-binary/Other / Unknown; originals in `Gender_raw` |
| `state` | "Non-US" for non-US respondents, "Unknown" for 11 US blanks |
| `Country_grouped` | New column: countries with <20 respondents grouped as "Other" |
| `self_employed` | 18 blanks (question added after launch) labeled "Unknown" |
| `work_interfere` | 264 blanks labeled "Unknown" (question only applies to people with a condition, but a blank doesn't confirm they have none) |
| `comments` | Dropped (87% empty free text) |
| `Timestamp` | Converted to datetime; 69 responses came in 2015–2016 and were kept |

**For modeling:** `work_interfere` nearly reveals `treatment`, so we need to decide as a team whether to include it.
**For analysis:** Non-binary/Other has only 14 people, which is too few for reliable stats on its own.

**For charts/modeling:** `no_employees`, `leave`, and `work_interfere` are ordered scales. A CSV can't store category order, so set it after loading or they'll sort alphabetically:

```python
size_order = ["1-5", "6-25", "26-100", "100-500", "500-1000", "More than 1000"]
df["no_employees"] = pd.Categorical(df["no_employees"], categories=size_order, ordered=True)
```